# FT -> XGB/ResNet two-step prediction

This notebook loads a saved FT model to generate embeddings,
then loads a saved XGB or ResNet model trained on those embeddings to predict new data.

Prereqs:
- Step 1 (FT embeddings) and Step 2 (XGB/ResNet) training already completed.
- FT output dir contains `model/01_<model>_FTTransformer.pth`.
- XGB output dir contains `model/01_<model>_Xgboost.pkl` (if using XGB).
- ResNet output dir contains `model/01_<model>_ResNet.pth` (if using ResNet).


In [ ]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / 'ins_pricing').exists():
    for parent in repo_root.parents:
        if (parent / 'ins_pricing').exists():
            repo_root = parent
            sys.path.insert(0, str(repo_root))
            break

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import json
import joblib
import pandas as pd
import torch

from ins_pricing.production.predict import load_predictor_from_config

In [ ]:
ft_cfg_path = repo_root / \
    'ins_pricing/examples/modelling/config_ft_unsupervised_ddp_embed.json'
xgb_cfg_path = repo_root / \
    'ins_pricing/examples/modelling/config_xgb_from_ft_unsupervised.json'
resn_cfg_path = repo_root / \
    'ins_pricing/examples/modelling/config_resn_from_ft_unsupervised_ddp.json'

# New data to score (must include the raw FT features).
input_path = repo_root / 'ins_pricing/examples/modelling/Data/od_bc.csv'

# Optional output file for predictions.
output_path = repo_root / 'ins_pricing/examples/modelling/Results/predictions_ft_xgb.csv'

# If configs contain multiple models, set model_name explicitly (e.g., 'od_bc').
model_name = None

# Models to score: choose any of ["xgb", "resn"].
model_keys = ["xgb", "resn"]

In [ ]:
ft_cfg = json.loads(ft_cfg_path.read_text(encoding='utf-8'))
xgb_cfg = (
    json.loads(xgb_cfg_path.read_text(encoding='utf-8'))
    if "xgb" in model_keys
    else None
)
resn_cfg = (
    json.loads(resn_cfg_path.read_text(encoding='utf-8'))
    if "resn" in model_keys
    else None
)

if model_name is None:
    model_list = list(ft_cfg.get('model_list') or [])
    model_categories = list(ft_cfg.get('model_categories') or [])
    if len(model_list) != 1 or len(model_categories) != 1:
        raise ValueError('Provide model_name when config has multiple models.')
    model_name = f"{model_list[0]}_{model_categories[0]}"

ft_output_dir = (ft_cfg_path.parent / ft_cfg['output_dir']).resolve()
xgb_output_dir = (
    (xgb_cfg_path.parent / xgb_cfg['output_dir']).resolve()
    if xgb_cfg is not None
    else None
)
ft_prefix = ft_cfg.get('ft_feature_prefix', 'ft_emb')
xgb_task_type = str(xgb_cfg.get('task_type', 'regression')
                    ) if xgb_cfg else None

if ft_cfg.get('geo_feature_nmes'):
    raise ValueError(
        'FT inference with geo tokens is not supported in this example.')

In [ ]:
# Step 1: load FT model and generate embeddings.
ft_model_path = ft_output_dir / 'model' / f"01_{model_name}_FTTransformer.pth"
ft_payload = torch.load(ft_model_path, map_location='cpu')
if isinstance(ft_payload, dict) and 'model' in ft_payload:
    ft_model = ft_payload['model']
else:
    ft_model = ft_payload

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if hasattr(ft_model, 'device'):
    ft_model.device = device
if hasattr(ft_model, 'to'):
    ft_model.to(device)
if hasattr(ft_model, 'ft'):
    ft_model.ft.to(device)

df_new = pd.read_csv(input_path)
emb = ft_model.predict(df_new, return_embedding=True)
emb_cols = [f"pred_{ft_prefix}_{i}" for i in range(emb.shape[1])]
df_with_emb = df_new.copy()
df_with_emb[emb_cols] = emb

result = df_with_emb.copy()

In [ ]:
# Step 2: load XGB model and predict on embedded features.
if "xgb" in model_keys:
    xgb_model_path = xgb_output_dir / 'model' / f"01_{model_name}_Xgboost.pkl"
    xgb_payload = joblib.load(xgb_model_path)
    if isinstance(xgb_payload, dict) and 'model' in xgb_payload:
        xgb_model = xgb_payload['model']
        feature_list = (
            xgb_payload.get('preprocess_artifacts', {})
            .get('factor_nmes')
        )
    else:
        xgb_model = xgb_payload
        feature_list = None

    if not feature_list:
        feature_list = xgb_cfg.get('feature_list') or []
    if not feature_list:
        raise ValueError('feature_list is missing; cannot build XGB input.')

    X = df_with_emb[feature_list]
    if xgb_task_type == 'classification' and hasattr(xgb_model, 'predict_proba'):
        pred = xgb_model.predict_proba(X)[:, 1]
    else:
        pred = xgb_model.predict(X)

    result['pred_xgb'] = pred

# Step 3: load ResNet model and predict on embedded features.
if "resn" in model_keys:
    resn_predictor = load_predictor_from_config(
        resn_cfg_path,
        "resn",
        model_name=model_name,
    )
    pred_resn = resn_predictor.predict(df_with_emb)
    result['pred_resn'] = pred_resn

if output_path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)

result.head()